# Rule-Based Anomaly Detector

This notebook implements the first anomaly-detection baseline for the CATS dataset.

The purpose of this detector is to establish a simple, interpretable baseline against which the more advanced rolling statistical and machine-learning detectors can later be compared.

The detector will identify observations that fall outside predefined normal-behaviour limits for the telemetry signals.

The thresholds will be derived from the nominal period only.

Ground-truth anomaly labels and metadata will not be used to determine the thresholds or generate predictions.

## Step 1 — Define the Detection Principle

A rule-based detector requires a definition of what constitutes an unusual observation.

Because the 17 telemetry signals have different numerical scales and distributions, one common absolute threshold cannot be applied to every signal.

Instead, each signal will have its own normal-behaviour limits derived from the nominal period.

An observation will be flagged when one or more signals fall outside their corresponding limits.

The detector is intended to be:

- simple;
- interpretable;
- reproducible;
- independent of the ground-truth anomaly labels;
- suitable as a baseline for comparison with later detectors.

## Step 2 — Establish Signal-Specific Thresholds

The rule-based detector will define a normal range separately for each telemetry signal.

For the initial baseline, the normal range will be defined using the 1st and 99th percentiles of each signal during the nominal period.

Observations below the 1st percentile or above the 99th percentile will be considered unusual for that signal.

The thresholds are calculated exclusively from the nominal period.

No anomaly labels or anomaly metadata are used to determine these thresholds.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"

df = pd.read_parquet(DATA_PATH)

nominal_df = df.iloc[:1_000_000].copy()

signal_columns = [
    column
    for column in nominal_df.columns
    if column not in ["y", "category"]
]

In [ ]:
rule_thresholds = nominal_df[signal_columns].quantile(
    [0.01, 0.99]
).T

rule_thresholds.columns = ["lower_threshold", "upper_threshold"]

print(rule_thresholds.round(3))